In [11]:
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import classification_report
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
with open("checkpoints/vocab.pkl", "rb") as f:
    data = pickle.load(f)

word2idx = {w.lower(): i for w, i in data["word2idx"].items()}
idx2word = {i: w.lower() for i, w in data["idx2word"].items()}
vocab = [w.lower() for w in data["vocab"]]

checkpoint = torch.load("checkpoints/checkpoint_19.pt", map_location="cpu")
embedding_matrix = checkpoint["model_state_dict"]["in_embeddings.weight"].detach().float().cpu()

vocab_size, embedding_dim = embedding_matrix.shape
print("Original pretrained matrix:", embedding_matrix.shape)

extra_tokens = []
if "<PAD>" not in word2idx:
    word2idx["<PAD>"] = len(word2idx)
    pad_vector = torch.zeros(1, embedding_dim)  
    extra_tokens.append(pad_vector)

if "<UNK>" not in word2idx:
    word2idx["<UNK>"] = len(word2idx)
    unk_vector = torch.randn(1, embedding_dim) * 0.01 
    extra_tokens.append(unk_vector)

if extra_tokens:
    extra_matrix = torch.cat(extra_tokens, dim=0)
    embedding_matrix = torch.cat([embedding_matrix, extra_matrix], dim=0)

print("Expanded embedding matrix:", embedding_matrix.shape)
PAD_IDX = word2idx["<PAD>"]



Original pretrained matrix: torch.Size([34154, 300])
Expanded embedding matrix: torch.Size([34156, 300])


In [13]:
dataset = load_dataset("batterydata/pos_tagging")
train_sentences = [[w.lower() for w in sent] for sent in dataset["train"]["words"]]
train_tags      = dataset["train"]["labels"]

test_sentences  = [[w.lower() for w in sent] for sent in dataset["test"]["words"]]
test_tags       = dataset["test"]["labels"]


tag_set = set([t for seq in train_tags for t in seq])
tag2idx = {t: i for i, t in enumerate(sorted(tag_set))}
idx2tag = {i: t for t, i in tag2idx.items()}

PAD_IDX = word2idx["<PAD>"]


In [14]:
class POSTaggingDataset(Dataset):
    def __init__(self, sentences, tags, word2idx, tag2idx, max_len=50):
        self.sentences = sentences
        self.tags = tags
        self.word2idx = word2idx
        self.tag2idx = tag2idx
        self.max_len = max_len
        self.pad_token_id = word2idx["<PAD>"]
        self.pad_tag_id = -100

    def encode_sentence(self, sentence):
        ids = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in sentence]
        return ids[:self.max_len] + [self.pad_token_id] * (self.max_len - len(ids))

    def encode_tags(self, tags):
        ids = [self.tag2idx[t] for t in tags]
        return ids[:self.max_len] + [self.pad_tag_id] * (self.max_len - len(ids))

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return torch.tensor(self.encode_sentence(self.sentences[idx])), \
               torch.tensor(self.encode_tags(self.tags[idx]))

train_data = POSTaggingDataset(train_sentences, train_tags, word2idx, tag2idx)
test_data  = POSTaggingDataset(test_sentences, test_tags, word2idx, tag2idx)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=64)


In [15]:
class BiLSTMPOSTagger(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=False, padding_idx=pad_idx
        )
        embed_dim = embedding_matrix.shape[1]
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

model = BiLSTMPOSTagger(
    embedding_matrix=embedding_matrix,
    hidden_dim=256,
    num_classes=len(tag2idx),
    pad_idx=PAD_IDX
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

unk_count = sum(word not in word2idx for sent in train_sentences for word in sent)
total_words = sum(len(sent) for sent in train_sentences)
print(f"UNK ratio: {unk_count/total_words:.2%}")

UNK ratio: 9.18%


In [16]:
def save_checkpoint(model, optimizer, epoch, path="POS_checkpoints"):
    os.makedirs(path, exist_ok=True)
    filename = os.path.join(path, f"POS_checkpoints_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict()
    }, filename)
    print(f"Saved checkpoint: {filename}")

def load_latest_checkpoint(model, optimizer, path="POS_checkpoints"):
    if not os.path.isdir(path):
        return 0, model, optimizer
    checkpoints = [f for f in os.listdir(path) if f.endswith(".pt")]
    if not checkpoints:
        return 0, model, optimizer
    latest = max(checkpoints, key=lambda f: int(f.replace("POS_checkpoints_", "").replace(".pt","")))
    ckpt = torch.load(os.path.join(path, latest), map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print(f"Loaded checkpoint: {latest}")
    return ckpt["epoch"], model, optimizer

start_epoch, model, optimizer = load_latest_checkpoint(model, optimizer)

EPOCHS = 15
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    for words, labels in train_loader:
        words, labels = words.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(words)
        loss = criterion(outputs.view(-1, outputs.shape[-1]), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")
    save_checkpoint(model, optimizer, epoch+1)


Loaded checkpoint: POS_checkpoints_15.pt


In [17]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for words, labels in test_loader:
        words, labels = words.to(device), labels.to(device)
        outputs = model(words)
        preds = outputs.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

valid_idx = [i for i,l in enumerate(all_labels) if l != -100]
all_preds = np.array(all_preds)[valid_idx]
all_labels = np.array(all_labels)[valid_idx]

unique_labels = sorted(set(all_labels))
target_names = [idx2tag[i] for i in unique_labels]

print(classification_report(all_labels, all_preds, labels=unique_labels, target_names=target_names, zero_division=0))


              precision    recall  f1-score   support

           #       1.00      1.00      1.00        15
           $       1.00      1.00      1.00       329
          ''       1.00      1.00      1.00       207
           ,       1.00      1.00      1.00      1780
       -LRB-       0.33      0.02      0.04        52
      -NONE-       0.95      0.82      0.88      2469
       -RRB-       0.88      0.14      0.24        51
           .       1.00      1.00      1.00      1424
           :       1.00      1.00      1.00       151
          CC       1.00      0.99      1.00       820
          CD       0.92      0.91      0.92      1706
          DT       0.99      0.99      0.99      2955
          EX       0.97      1.00      0.98        29
          FW       0.00      0.00      0.00         1
          IN       0.97      0.98      0.98      3680
          JJ       0.79      0.86      0.83      2157
         JJR       0.94      0.78      0.85       186
         JJS       0.94    

In [18]:
import torch.nn.functional as F

def predict_sentence(model, sentence, word2idx, idx2tag, max_len=50):
    model.eval()
    sentence_orig = sentence[:]               
    sentence = [w.lower() for w in sentence]  
    ids = [word2idx.get(w, word2idx["<UNK>"]) for w in sentence]
    ids = ids[:max_len] + [word2idx["<PAD>"]] * (max_len - len(ids))
    x = torch.tensor([ids]).to(device)

    with torch.no_grad():
        outputs = model(x)   
        probs = F.softmax(outputs, dim=-1)  
        preds = probs.argmax(dim=-1).cpu().numpy()[0]
        confs = probs.max(dim=-1).values.cpu().numpy()[0]

    results = []
    for orig, p, c in zip(sentence_orig, preds[:len(sentence_orig)], confs[:len(sentence_orig)]):
        results.append((orig, idx2tag[p], round(float(c), 4)))
    return results

print(predict_sentence(model, ["The", "dog", "runs", "fast"], word2idx, idx2tag))


[('The', 'DT', 0.9994), ('dog', 'NN', 0.9886), ('runs', 'VBZ', 0.9941), ('fast', 'RB', 0.9992)]


In [25]:
test_sentences = [
    ["The", "dog", "runs", "football"],
    ["I", "will","be", "singing", "today"],
]

for sent in test_sentences:
    preds = predict_sentence(model, sent, word2idx, idx2tag)
    print(preds)


[('The', 'DT', 0.9991), ('dog', 'NN', 0.9269), ('runs', 'VBZ', 0.9971), ('football', 'NN', 0.9418)]
[('I', 'PRP', 0.9988), ('will', 'MD', 0.9998), ('be', 'VB', 0.9998), ('singing', 'VBN', 0.9663), ('today', 'NN', 0.9995)]
